# 02 — RFM & Cohort Analysis

**Goal:** Build the full RFM feature set and cohort retention matrix that will feed the BG/NBD model in Phase 2.

**Tools:** Polars (lazy pipeline), DuckDB (SQL aggregations), Plotly (charts)  
**Outputs:** `rfm_features` and `customers` rows in Supabase.

---

### Sections
1. Load and split data (calibration / holdout)
2. Compute RFM features with Polars
3. Compute cohort features
4. Build ground-truth LTV labels from holdout
5. Cohort retention matrix
6. LTV development over time by cohort
7. W&B run tracking
8. Save to Supabase


In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import polars as pl
from IPython.display import display

sys.path.insert(0, str(Path().resolve().parent))

from backend.config import settings
from backend.data.load_data import load_uci_csv
from backend.features.rfm import (
    clean_transactions,
    assign_product_categories,
    assign_amount_buckets,
    make_calibration_holdout_split,
    RFMPipeline,
)
from backend.features.cohorts import CohortPipeline
from backend.features.duckdb_agg import DuckDBAggregator
from backend.db.supabase_client import SupabaseClient

import plotly.io as pio
pio.templates.default = 'plotly_white'

print('✓ Imports OK')

In [ ]:
# ── Config ────────────────────────────────────────────────────────────
OBSERVATION_MONTHS = 6
HOLDOUT_MONTHS     = 6
SAVE_TO_DB         = True   # set False to skip Supabase writes
USE_WANDB          = True   # set False to skip W&B logging

print(f'Observation window: {OBSERVATION_MONTHS} months')
print(f'Holdout window:     {HOLDOUT_MONTHS} months')

## 1. Load and Split Data

In [ ]:
# Load raw and clean
raw_df  = load_uci_csv(settings.UCI_CSV_PATH)
cleaned = clean_transactions(raw_df)
cleaned = assign_product_categories(cleaned)
cleaned = assign_amount_buckets(cleaned)

print(f'Cleaned transactions: {len(cleaned):,}')

# Calibration / holdout split
calibration, holdout, obs_end, holdout_end = make_calibration_holdout_split(
    cleaned,
    observation_months=OBSERVATION_MONTHS,
    holdout_months=HOLDOUT_MONTHS,
)

print(f'\nCalibration: {len(calibration):,} rows  (up to {obs_end})')
print(f'Holdout:     {len(holdout):,} rows  ({obs_end} – {holdout_end})')
print(f'\nCalibration customers: {calibration["customer_id"].n_unique():,}')
print(f'Holdout customers:     {holdout["customer_id"].n_unique():,}')

In [ ]:
# Quick split sanity check
fig = make_subplots(rows=1, cols=2, subplot_titles=['Calibration', 'Holdout'])

for i, (df, label) in enumerate([(calibration, 'Calibration'), (holdout, 'Holdout')], start=1):
    monthly = (
        df.with_columns(pl.col('invoice_date').dt.strftime('%Y-%m').alias('month'))
          .group_by('month').agg((pl.col('quantity') * pl.col('unit_price')).sum().alias('rev'))
          .sort('month')
    )
    fig.add_trace(go.Bar(x=monthly['month'].to_list(), y=monthly['rev'].to_list(), name=label), row=1, col=i)

fig.update_layout(height=350, title='Monthly Revenue: Calibration vs Holdout', showlegend=False)
fig.show()

## 2. Compute RFM Features

In [ ]:
# Run the RFM pipeline on calibration data
rfm_pipeline = RFMPipeline(
    transactions=calibration,
    observation_end_date=obs_end,
)

rfm_df = rfm_pipeline.compute()
print(f'RFM computed for {len(rfm_df):,} customers')
print(f'Columns: {rfm_df.columns}')

In [ ]:
# Preview
display(rfm_df.select([
    'customer_id', 'frequency', 'recency_days', 't_days',
    'monetary_avg', 'monetary_total', 'orders_count',
    'cohort_month', 'days_to_second_purchase'
]).head(10))

In [ ]:
# Summary statistics — the ones BG/NBD needs most
bgnbd_cols = ['frequency', 'recency_days', 't_days', 'monetary_avg']
print('=== BG/NBD Input Statistics ===')
display(rfm_df.select(bgnbd_cols).describe())

# How many customers have frequency > 0 (repeat buyers)?
repeat = (rfm_df['frequency'] > 0).sum()
print(f'\nRepeat buyers (freq>0): {repeat:,} / {len(rfm_df):,}  ({100*repeat/len(rfm_df):.1f}%)')

In [ ]:
# Calibration check: frequency vs recency (BG/NBD standard diagnostic)
# Customers with higher frequency should generally have higher recency
fig = px.scatter(
    rfm_df.filter(pl.col('frequency') <= 30).to_pandas(),
    x='t_days', y='recency_days',
    color='frequency',
    color_continuous_scale='Viridis',
    opacity=0.5,
    title='T vs Recency (coloured by Frequency) — BG/NBD Calibration Diagnostic',
    labels={'t_days': 'Customer Age T (days)', 'recency_days': 'Recency (days)'},
)
# Add T = recency line (upper bound)
max_t = rfm_df['t_days'].max()
fig.add_trace(go.Scatter(x=[0, max_t], y=[0, max_t], mode='lines',
                          line=dict(color='red', dash='dash'), name='T = recency'))
fig.show()

In [ ]:
# Days to second purchase — strong churn predictor
d2s = rfm_df.filter(pl.col('days_to_second_purchase').is_not_null())
d2s_capped = d2s.filter(pl.col('days_to_second_purchase') <= 120)

fig = px.histogram(
    d2s_capped.to_pandas(),
    x='days_to_second_purchase', nbins=60,
    title='Days to Second Purchase (≤120 days)',
    labels={'days_to_second_purchase': 'Days to 2nd Purchase'},
)
# Add 30-day vertical line
fig.add_vline(x=30, line_dash='dash', line_color='red',
              annotation_text='30 days', annotation_position='top right')
fig.show()

fast_return = (d2s['days_to_second_purchase'] <= 30).sum()
print(f'Returned within 30 days: {fast_return:,} / {len(d2s):,} ({100*fast_return/len(d2s):.1f}%)')

## 3. Cohort Features

In [ ]:
cohort_pipeline = CohortPipeline(calibration)

# Cohort assignments
cohort_df = cohort_pipeline.compute_cohort_assignments()
cohort_sizes = cohort_pipeline.compute_cohort_sizes(cohort_df)

print(f'{len(cohort_df)} customers across {cohort_df["cohort_month"].n_unique()} monthly cohorts')
display(cohort_sizes)

In [ ]:
# Cohort sizes over time
fig = px.bar(
    cohort_sizes.to_pandas(),
    x='cohort_month', y='cohort_size',
    title='Monthly Cohort Acquisition Size',
    labels={'cohort_month': 'Cohort Month', 'cohort_size': 'New Customers'},
    color='cohort_size', color_continuous_scale='Blues',
)
fig.show()

In [ ]:
# Enrich RFM with cohort metadata
rfm_enriched = cohort_pipeline.enrich_rfm(rfm_df, cohort_df, obs_end)
print(f'Enriched RFM columns: {rfm_enriched.columns}')

## 4. Ground-Truth LTV Labels

In [ ]:
# Attach 12m LTV labels from holdout
rfm_labelled = rfm_pipeline.compute_ltv_labels(
    holdout_df=holdout,
    rfm_df=rfm_enriched,
    horizon_months=12,
)

n_nonzero = (rfm_labelled['actual_ltv_12m'] > 0).sum()
print(f'Customers with >0 LTV in holdout: {n_nonzero:,} / {len(rfm_labelled):,}  ({100*n_nonzero/len(rfm_labelled):.1f}%)')
print(f'\nActual 12m LTV distribution:')
print(rfm_labelled.filter(pl.col('actual_ltv_12m') > 0)['actual_ltv_12m'].describe())

In [ ]:
# LTV distribution
ltv_pos = rfm_labelled.filter(pl.col('actual_ltv_12m') > 0)
p99 = ltv_pos['actual_ltv_12m'].quantile(0.99)

fig = px.histogram(
    ltv_pos.filter(pl.col('actual_ltv_12m') <= p99).to_pandas(),
    x='actual_ltv_12m', nbins=80,
    title=f'Actual 12m LTV Distribution (positive buyers, ≤ £{p99:.0f} p99)',
    labels={'actual_ltv_12m': 'Actual 12m LTV (£)'},
)
fig.show()

# How skewed?
mean_ltv   = ltv_pos['actual_ltv_12m'].mean()
median_ltv = ltv_pos['actual_ltv_12m'].median()
print(f'Mean LTV:   £{mean_ltv:.2f}')
print(f'Median LTV: £{median_ltv:.2f}')
print(f'Skew ratio (mean/median): {mean_ltv/median_ltv:.2f}x  — highly right-skewed, Huber loss needed')

In [ ]:
# Frequency vs actual LTV — key validation
sample = rfm_labelled.filter(
    (pl.col('actual_ltv_12m') > 0) &
    (pl.col('actual_ltv_12m') < rfm_labelled['actual_ltv_12m'].quantile(0.98))
).sample(min(2000, len(rfm_labelled)))

fig = px.scatter(
    sample.to_pandas(),
    x='frequency', y='actual_ltv_12m',
    color='monetary_avg',
    color_continuous_scale='RdYlGn',
    opacity=0.5,
    trendline='ols',
    title='Frequency vs Actual 12m LTV (coloured by avg order value)',
    labels={'frequency': 'Calibration Frequency', 'actual_ltv_12m': 'Actual 12m LTV (£)'},
)
fig.show()

## 5. Cohort Retention Matrix

In [ ]:
# Full retention matrix (calibration + holdout combined)
cohort_pipeline_full = CohortPipeline(cleaned)  # use all data
cohort_df_full = cohort_pipeline_full.compute_cohort_assignments()
retention = cohort_pipeline_full.compute_retention_matrix(cohort_df_full, max_months=11)

print(f'Retention matrix: {len(retention)} rows')
retention.head(10)

In [ ]:
# Pivot to wide format for heatmap
retention_pivot = retention.pivot(
    values='retention_rate_pct',
    index='cohort_month',
    on='months_since_first',
    aggregate_function='first',
).sort('cohort_month')

cohorts = retention_pivot['cohort_month'].to_list()
month_cols = [c for c in retention_pivot.columns if c != 'cohort_month']
z = retention_pivot.select(month_cols).to_numpy()

fig = go.Figure(go.Heatmap(
    z=z,
    x=[f'M+{m}' for m in month_cols],
    y=cohorts,
    colorscale='RdYlGn',
    zmin=0, zmax=100,
    text=[[f'{v:.0f}%' if v is not None else '' for v in row] for row in z.tolist()],
    texttemplate='%{text}',
    textfont=dict(size=10),
    colorbar=dict(title='Retention %'),
))
fig.update_layout(
    title='Cohort Retention Matrix',
    xaxis_title='Months Since First Purchase',
    yaxis_title='Acquisition Cohort',
    height=450,
)
fig.show()

## 6. LTV Development Over Time by Cohort

In [ ]:
ltv_curves = cohort_pipeline_full.compute_cohort_ltv_over_time(
    cohort_df_full, max_months=11
)
print(f'LTV curve rows: {len(ltv_curves)}')

fig = px.line(
    ltv_curves.to_pandas(),
    x='months_since_first',
    y='cumulative_ltv_per_customer',
    color='cohort_month',
    title='Cumulative LTV per Customer by Cohort',
    labels={
        'months_since_first': 'Months Since First Purchase',
        'cumulative_ltv_per_customer': 'Cumulative LTV / Customer (£)',
    },
)
fig.show()

## 7. W&B Logging

In [ ]:
if USE_WANDB:
    try:
        import wandb
        
        run = wandb.init(
            project=settings.WANDB_PROJECT,
            name='week1_rfm_cohort',
            tags=['week1', 'rfm', 'feature_engineering'],
            config={
                'observation_months':    OBSERVATION_MONTHS,
                'holdout_months':        HOLDOUT_MONTHS,
                'observation_end':       str(obs_end),
                'holdout_end':           str(holdout_end),
                'n_raw_rows':            len(raw_df),
                'n_cleaned_rows':        len(cleaned),
                'n_calibration_rows':    len(calibration),
                'n_holdout_rows':        len(holdout),
                'n_customers':           len(rfm_labelled),
                'n_repeat_buyers':       int((rfm_labelled['frequency'] > 0).sum()),
                'pct_nonzero_ltv_12m':   round(100 * n_nonzero / len(rfm_labelled), 2),
            }
        )
        
        # Log summary stats
        wandb.log({
            'mean_ltv_12m':           float(rfm_labelled.filter(pl.col('actual_ltv_12m') > 0)['actual_ltv_12m'].mean()),
            'median_ltv_12m':         float(rfm_labelled.filter(pl.col('actual_ltv_12m') > 0)['actual_ltv_12m'].median()),
            'mean_frequency':         float(rfm_labelled['frequency'].mean()),
            'mean_monetary_avg':      float(rfm_labelled['monetary_avg'].mean()),
            'n_customers':            len(rfm_labelled),
            'n_cohorts':              rfm_labelled['cohort_month'].n_unique(),
        })
        
        # Log RFM table
        rfm_sample = rfm_labelled.sample(min(500, len(rfm_labelled)))
        wandb.log({'rfm_sample': wandb.Table(dataframe=rfm_sample.to_pandas())})
        
        wandb.finish()
        print('✓ W&B run logged')
    except Exception as e:
        print(f'W&B logging skipped: {e}')
else:
    print('W&B logging disabled (USE_WANDB=False)')

## 8. Save to Supabase

In [ ]:
if SAVE_TO_DB:
    db = SupabaseClient(use_service_role=True)
    assert db.health_check(), 'DB health check failed — check .env'
    
    print('Saving RFM features to Supabase...')
    n_saved = rfm_pipeline.save(rfm_labelled, db)
    print(f'✓ Saved {n_saved:,} RFM rows')
else:
    print('Skipping DB save (SAVE_TO_DB=False)')
    print(f'\nRFM DataFrame ready with {len(rfm_labelled):,} rows')

In [ ]:
# Final summary
print('\n=== Week 1 Complete — Summary ===')
print(f'  Calibration customers:     {len(rfm_labelled):,}')
print(f'  Observation window:        up to {obs_end}')
print(f'  Holdout window:            {obs_end} – {holdout_end}')
print(f'  Repeat buyers (freq > 0):  {int((rfm_labelled["frequency"] > 0).sum()):,}')
print(f'  Holdout LTV coverage:      {100*n_nonzero/len(rfm_labelled):.1f}% have >0 actual LTV')
print(f'  Mean actual 12m LTV:       £{rfm_labelled.filter(pl.col("actual_ltv_12m")>0)["actual_ltv_12m"].mean():.2f}')
print(f'  Cohort retention saved:    {len(retention)} matrix rows')
print(f'\nNext: notebooks/03_bgnbd_gamma_gamma.ipynb (Phase 2)')